In [21]:
import pandas as pd
import os

In [22]:

query = "adbl"
newspath = f"../DATA-HTML-STOCK/STOCKSENTIMENT/{query}_sentiment_results.csv"
sentiment_score = 0.00
if os.path.exists(newspath):
    sentiment_df = pd.read_csv(f"{newspath}")
    sentiment_plus_datas = sentiment_df[['Date', 'Sentiment_Score', 'Prediction']]
    print(sentiment_plus_datas.head())
else:
    
    print(f"no news {query} file found")


# sentiment_df["Sentiment_Score"] = sentiment_df["Sentiment_Score"].astype(float)

         Date  Sentiment_Score Prediction
0  2026-02-17         0.953698   positive
1  2026-02-12         0.829293   positive
2  2026-01-14        -0.913054   negative
3  2025-12-15        -0.884442   negative
4  2025-12-11        -0.604346   negative


In [23]:
stockpath = f"../DATA-HTML-STOCK/NEPSEDATA/{query}.csv"

if os.path.exists(stockpath):
    stock_df = pd.read_csv(f"{stockpath}")
    stock_close_plus_date = stock_df[["Date", "Close"]]
    print(stock_close_plus_date.head())
else:
    print(f"no stock {query} found")
    print(f"Prediction file of {query} ignored")
    # break



         Date   Close
0  2026-02-24  293.00
1  2026-02-23  292.50
2  2026-02-22  293.00
3  2026-02-17  296.50
4  2026-02-16  297.00


In [27]:
stock_df = stock_close_plus_date
news_df = sentiment_plus_datas

stock_df = stock_df.rename(columns={"Date": "Date_Stock"})
news_df = news_df.rename(columns={"Date": "Date_News"})

news_df = news_df.groupby("Date_News").agg({
    "Sentiment_Score": "mean"
}).reset_index()

news_df["Prediction"] = news_df["Sentiment_Score"].apply(
    lambda x: "Positive" if x > 0 
    else "Negative" if x < 0 
    else "Neutral"
)

df = pd.merge(stock_df, news_df, left_on="Date_Stock", right_on="Date_News", how="left")

df["Sentiment_Score"] = df["Sentiment_Score"].fillna(0)
df["Prediction"] = df["Prediction"].fillna("Neutral")
df["Close"] = stock_close_plus_date['Close']

df = df[["Date_Stock", "Close", "Date_News", "Sentiment_Score", "Prediction"]]

print(df)

      Date_Stock   Close   Date_News  Sentiment_Score Prediction
0     2026-02-24  293.00         NaN         0.000000    Neutral
1     2026-02-23  292.50         NaN         0.000000    Neutral
2     2026-02-22  293.00         NaN         0.000000    Neutral
3     2026-02-17  296.50  2026-02-17         0.953698   Positive
4     2026-02-16  297.00         NaN         0.000000    Neutral
...          ...     ...         ...              ...        ...
3519  2010-09-12  118.00         NaN         0.000000    Neutral
3520  2010-09-09  122.00         NaN         0.000000    Neutral
3521  2010-09-08  125.00         NaN         0.000000    Neutral
3522  2010-09-07  138.00         NaN         0.000000    Neutral
3523  2010-09-02  255.00         NaN         0.000000    Neutral

[3524 rows x 5 columns]


In [28]:
savepath = f"../DATA-HTML-STOCK/FinalDataset/{query.upper()}.csv"
df.to_csv(savepath, index=False)